# 6.3 — Activation Functions

Activation functions turn raw affine scores into usable neural-network signals: some gates pass positive evidence, some squeeze values into bounded ranges, and softmax turns competing logits into probabilities. In this lesson, you will build sigmoid, tanh, ReLU, GELU, and softmax from scratch in NumPy, then inspect how their local arithmetic controls gradient flow, numerical scale, and practical training behavior.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build activation functions one idea at a time. Run each cell in order and read the printed intermediate values — every piece of math is shown so the activation is not a black box. This walkthrough is self-contained (it imports what it needs) and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, exponentials, and vectorized activation math.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for every small numerical demo.

### 1. Affine pre-activation: the raw signal before any nonlinearity

A neuron first computes an affine score $z=w^\top x+b$. This is only a weighted sum plus a shift, so by itself a stack of layers would still collapse into one linear map. The activation function is the next step that decides how this raw score should be reshaped before later layers see it.

In [ ]:
x_w = np.array([1.5, -0.5])          # two input features from the lesson block.
w_w = np.array([1.6, 0.2])           # two weights controlling feature influence.
b_w = 0.6                            # bias shifts the decision threshold.
parts_w = w_w * x_w                  # coordinate contributions to w^T x.
z_w = float(parts_w.sum() + b_w)     # affine pre-activation.
print("weighted parts:", parts_w)    # 1.6*1.5 and 0.2*(-0.5).
print("z =", round(z_w, 3))          # 2.4 - 0.1 + 0.6 = 2.9.
assert round(z_w, 3) == 2.900

▶ What you'll see: the two feature contributions are `2.4` and `-0.1`, and the bias raises the final pre-activation to `2.9`.

In [ ]:
plt.figure(figsize=(4.4, 3))
plt.bar(["w0*x0", "w1*x1", "bias", "z"], [parts_w[0], parts_w[1], b_w, z_w],
        color=["seagreen", "indianred", "gray", "steelblue"])
plt.axhline(0, color="black", linewidth=0.8)
plt.title("1: affine signal before activation")
plt.ylabel("value")
plt.show()

▶ What you'll see: one feature pushes the score up, one pushes it down, and the bias sets the operating point.

*Why it's done this way:* the affine score collects evidence on a real-valued scale, but it has no built-in notion of gating, saturation, or probability. Separating $z=w^\top x+b$ from $h=\phi(z)$ lets us reason about two different jobs: weights decide what evidence is present, while the activation decides how that evidence is allowed to pass forward.

### 2. ReLU: a hard gate for positive evidence

ReLU uses $\phi(z)=\max(0,z)$. Positive scores pass through unchanged; negative scores become exactly zero. That makes it cheap, sparse, and easy to differentiate almost everywhere, but it also means a very negative unit can stop sending gradient through its input side.

In [ ]:
z_grid_w = np.linspace(-4, 4, 9)       # inspect ReLU at evenly spaced pre-activations.
relu_w = np.maximum(0, z_grid_w)       # hard gate: negatives become 0, positives pass.
relu_grad_w = (z_grid_w > 0).astype(float)  # derivative is 0 left of zero and 1 right of zero.
print("z:", z_grid_w)
print("ReLU(z):", relu_w)
print("ReLU'(z):", relu_grad_w)
assert relu_w[4] == 0 and relu_w[-1] == 4

▶ What you'll see: all negative inputs output `0`, while positive inputs are copied exactly.

In [ ]:
h_w = float(np.maximum(0, z_w))        # apply ReLU to the lesson score 2.9.
print("ReLU(2.9) =", round(h_w, 3))    # positive score passes unchanged.
assert round(h_w, 3) == 2.900
plt.figure(figsize=(4.6, 3))
plt.plot(z_grid_w, relu_w, marker="o", label="ReLU")
plt.plot(z_grid_w, relu_grad_w, marker="s", label="ReLU derivative")
plt.axvline(0, color="black", linestyle="--", linewidth=0.8)
plt.title("2: ReLU gate and derivative")
plt.legend()
plt.show()

▶ What you'll see: a kink at zero — the output turns on there, and the derivative jumps from 0 to 1.

*Why it's done this way:* ReLU preserves scale for active units, so gradients do not shrink just because the activation is positive. The zero side is the constraint: it removes negative evidence and creates sparse activations, which often helps optimization, but the same zero derivative explains the dead-ReLU pitfall.

### 3. Sigmoid and tanh: bounded activations and saturation

Sigmoid maps to $(0,1)$, while tanh maps to $(-1,1)$. They are useful when a bounded signal is meaningful, but their derivatives become tiny in the tails. Tiny derivatives are the numerical reason deep chains can lose gradient signal.

In [ ]:
z_sat_w = np.array([-6., -2., 0., 2., 6.])       # values spanning both saturation tails.
sig_w = 1 / (1 + np.exp(-z_sat_w))               # sigmoid from scratch.
tanh_w = np.tanh(z_sat_w)                        # tanh is NumPy's stable hyperbolic tangent.
sig_grad_w = sig_w * (1 - sig_w)                 # derivative of sigmoid.
tanh_grad_w = 1 - tanh_w ** 2                    # derivative of tanh.
print("sigmoid:", np.round(sig_w, 3))
print("sigmoid grad:", np.round(sig_grad_w, 3))
print("tanh grad:", np.round(tanh_grad_w, 3))
assert round(float(sig_w[2]), 3) == 0.500 and round(float(sig_grad_w[2]), 3) == 0.250

▶ What you'll see: sigmoid's largest derivative is `0.25` at zero, and both derivatives shrink near ±6.

In [ ]:
z_dense_w = np.linspace(-6, 6, 200)
sig_dense_w = 1 / (1 + np.exp(-z_dense_w))
tanh_dense_w = np.tanh(z_dense_w)
plt.figure(figsize=(5, 3))
plt.plot(z_dense_w, sig_dense_w, label="sigmoid")
plt.plot(z_dense_w, tanh_dense_w, label="tanh")
plt.axhline(0, color="black", linewidth=0.6)
plt.title("3: bounded activations saturate")
plt.legend()
plt.show()

▶ What you'll see: both curves flatten in the tails, which is exactly where their gradients become small.

*Why it's done this way:* bounded activations deliberately limit the signal range, which can encode probabilities or centered states. The cost is saturation: when the curve is flat, changing $z$ barely changes $h$, and backpropagation multiplies by that small slope.

### 4. GELU: a smooth probabilistic gate

GELU behaves like a softened ReLU: instead of abruptly keeping positive values and killing negative values, it multiplies $z$ by a smooth gate. A common approximation is $\mathrm{GELU}(z)=0.5z\left(1+\tanh\left(\sqrt{2/\pi}(z+0.044715z^3)\right)\right)$.

In [ ]:
def gelu_w(z):
    return 0.5 * z * (1 + np.tanh(np.sqrt(2 / np.pi) * (z + 0.044715 * z ** 3)))

z_gelu_w = np.array([-3., -1., 0., 1., 3.])
relu_gelu_w = np.maximum(0, z_gelu_w)
gelu_vals_w = gelu_w(z_gelu_w)
print("ReLU:", np.round(relu_gelu_w, 3))
print("GELU:", np.round(gelu_vals_w, 3))
assert round(float(gelu_vals_w[3]), 3) == 0.841

▶ What you'll see: GELU keeps most of `1` but only a tiny negative amount of `-1`, rather than hard-zeroing it.

In [ ]:
z_curve_w = np.linspace(-4, 4, 300)
plt.figure(figsize=(5, 3))
plt.plot(z_curve_w, np.maximum(0, z_curve_w), label="ReLU", color="gray")
plt.plot(z_curve_w, gelu_w(z_curve_w), label="GELU", color="purple")
plt.axhline(0, color="black", linewidth=0.6)
plt.title("4: GELU is a smooth gate")
plt.legend()
plt.show()

▶ What you'll see: GELU bends smoothly through zero instead of creating ReLU's sharp corner.

*Why it's done this way:* the smooth gate can preserve small negative information and gives a smoother derivative for optimization. It still acts like a gate, but the transition is gradual, which is why modern deep architectures often prefer GELU-like activations.

### 5. Softmax: turn competing logits into probabilities

Softmax maps a vector of logits to positive probabilities that sum to one. It cares about relative differences, not absolute offsets. For numerical stability, we subtract the maximum logit before exponentiating; this leaves the probabilities unchanged while avoiding huge exponentials.

In [ ]:
logits_w = np.array([2.9, 0.4])                 # lesson comparison: score vs baseline.
exp_raw_w = np.exp(logits_w)                    # direct exponentials for visible arithmetic.
prob_raw_w = exp_raw_w / exp_raw_w.sum()        # direct softmax.
print("exp(logits):", np.round(exp_raw_w, 3))
print("softmax:", np.round(prob_raw_w, 3))
assert round(float(prob_raw_w[0]), 3) == 0.924

▶ What you'll see: `exp(2.9)=18.174`, `exp(0.4)=1.492`, so the first class receives probability about `0.924`.

In [ ]:
shifted_w = logits_w - np.max(logits_w)         # stable shift by the max logit.
prob_stable_w = np.exp(shifted_w) / np.exp(shifted_w).sum()
print("shifted logits:", shifted_w)
print("stable softmax:", np.round(prob_stable_w, 3))
assert np.allclose(prob_raw_w, prob_stable_w)
plt.figure(figsize=(4.2, 3))
plt.bar(["class 0", "class 1"], prob_stable_w, color=["seagreen", "gray"])
plt.ylim(0, 1)
plt.title("5: softmax probabilities sum to 1")
plt.show()

▶ What you'll see: the stable and direct probabilities match, but the shifted logits are safer to exponentiate.

*Why it's done this way:* exponentials convert score gaps into multiplicative odds, and normalization turns those odds into probabilities. Subtracting the max exploits the identity $\mathrm{softmax}(z)=\mathrm{softmax}(z-c)$, so the math is the same but the computer avoids overflow.

### 6. Activation derivatives: how local choices scale backpropagation

Backpropagation multiplies upstream gradient by the local derivative $\phi'(z)$. This means an activation is not just a forward transform; it is also a gradient valve. ReLU passes gradient for active units, sigmoid attenuates it, and saturated sigmoid nearly blocks it.

In [ ]:
upstream_w = 1.65                         # lesson gradient magnitude entering a scalar parameter/update demo.
z_back_w = np.array([-4., 0., 2.9])       # saturated, central, and active-positive examples.
sig_back_w = 1 / (1 + np.exp(-z_back_w))
sig_deriv_w = sig_back_w * (1 - sig_back_w)
relu_deriv_w = (z_back_w > 0).astype(float)
print("sigmoid local grads:", np.round(sig_deriv_w, 4))
print("ReLU local grads:", relu_deriv_w)

▶ What you'll see: sigmoid at `-4` has a tiny local gradient, while ReLU at `2.9` passes the upstream gradient unchanged.

In [ ]:
back_sig_w = upstream_w * sig_deriv_w
back_relu_w = upstream_w * relu_deriv_w
print("back through sigmoid:", np.round(back_sig_w, 4))
print("back through ReLU:", np.round(back_relu_w, 4))
eta_w = 0.08
theta_old_w = 2.0
theta_new_w = theta_old_w - eta_w * upstream_w
print("parameter update:", round(theta_new_w, 3))
assert round(theta_new_w, 3) == 1.868

▶ What you'll see: the same upstream gradient becomes much smaller through saturated sigmoid, while the scalar update moves `2.000` to `1.868`.

*Why it's done this way:* training is repeated reliable nudging. The learning rate controls step size, but the activation derivative controls how much of the error signal reaches earlier computations; bad local slopes can make a globally correct objective hard to optimize.

### 7. Scale, normalization, and memory bookkeeping

Activations live inside systems with finite numeric range and memory. Normalization asks whether a signal is unusual relative to a batch or layer scale, while memory accounting reminds us that saved activations are needed for backpropagation and can dominate hardware cost.

In [ ]:
z_norm_w = 2.9
mean_w = 1.0
var_w = 0.25
eps_w = 1e-5
normalized_w = (z_norm_w - mean_w) / np.sqrt(var_w + eps_w)
print("normalized activation:", round(normalized_w, 3))
assert round(normalized_w, 3) == 3.800

▶ What you'll see: the value `2.9` is `3.8` standard deviations above a mean of `1.0` with variance `0.25`.

In [ ]:
vectors_w = 5
width_w = 128
bytes_per_float_w = 4
memory_kb_w = vectors_w * width_w * bytes_per_float_w / 1024
print("activation memory KB:", round(memory_kb_w, 3))
assert round(memory_kb_w, 3) == 2.500
plt.figure(figsize=(4.5, 3))
plt.bar(["normalized z", "memory KB"], [normalized_w, memory_kb_w], color=["steelblue", "darkorange"])
plt.title("7: activation scale and storage")
plt.show()

▶ What you'll see: a numeric scale diagnostic beside a small memory calculation; both become large concerns in deep models.

*Why it's done this way:* normalization changes the effective scale seen by downstream activations and gradients, while saved activations are the data backpropagation must reuse. Activation math is therefore both calculus and systems bookkeeping.

## 🛠️ Setup

In [ ]:
import numpy as np # load NumPy for arrays, exponentials, vectorized activation functions, and assertions.
import matplotlib.pyplot as plt # load Matplotlib for activation curves, bars, heatmaps, and gradient diagnostics.
np.random.seed(0) # make the examples reproducible across notebook runs.

## 🟢 Basics (warm-up)

### Basic 1 — Compute an affine pre-activation

**Goal.** Build $z=w^\top x+b$ from two inputs, because every activation starts by reshaping a raw affine score. We build it in 2 steps.

In [ ]:
x_b1 = np.array([1.5, -0.5]) # define two input features for the tiny neuron.
w_b1 = np.array([1.6, 0.2]) # define one weight per feature.
b_b1 = 0.6 # define the bias term that shifts the score.
print("x:", x_b1, "w:", w_b1, "b:", b_b1) # inspect all affine ingredients.

▶ What you'll see: the neuron has two inputs, two weights, and one bias.

In [ ]:
parts_b1 = x_b1 * w_b1 # multiply each feature by its weight.
z_b1 = float(parts_b1.sum() + b_b1) # add contributions and bias to get the pre-activation.
print("parts:", parts_b1, "z:", round(z_b1, 3)) # inspect the arithmetic.
assert round(z_b1, 3) == 2.900 # verify the lesson's scratch-pass number.
plt.figure(figsize=(4, 3)) # create a compact contribution chart.
plt.bar(["x0w0", "x1w1", "b", "z"], [parts_b1[0], parts_b1[1], b_b1, z_b1], color="teal") # show the affine pieces.
plt.axhline(0, color="black", linewidth=0.8) # add a zero reference.
plt.title("Basic 1: affine pre-activation") # title the plot.
plt.show() # display the bar chart.

▶ What you'll see: the positive first feature dominates the score and the final z is 2.9.

👀 Takeaway: activations transform $z$, so always inspect the affine score first.

### Basic 2 — Apply ReLU to one score

**Goal.** Gate a single pre-activation with $\max(0,z)$, because ReLU is the simplest way to pass only positive evidence. We build it in 2 steps.

In [ ]:
z_b2 = 2.9 # reuse the lesson's pre-activation score.
h_b2 = np.maximum(0, z_b2) # apply ReLU to the scalar score.
print("z:", z_b2, "ReLU(z):", h_b2) # inspect the gated signal.
assert round(float(h_b2), 3) == 2.900 # positive values pass unchanged.

▶ What you'll see: ReLU leaves the positive score at 2.9.

In [ ]:
z_line_b2 = np.linspace(-3, 3, 100) # create inputs for a visible ReLU curve.
h_line_b2 = np.maximum(0, z_line_b2) # compute ReLU for every input.
plt.figure(figsize=(4, 3)) # create the curve plot.
plt.plot(z_line_b2, h_line_b2, color="seagreen") # draw the gate.
plt.axvline(0, color="black", linestyle="--", linewidth=0.8) # mark the threshold.
plt.title("Basic 2: ReLU(z)=max(0,z)") # title the plot.
plt.show() # display the curve.

▶ What you'll see: the curve is flat at 0 for negative z and linear for positive z.

👀 Takeaway: ReLU is a hard threshold followed by identity on the positive side.

### Basic 3 — Visualize ReLU's derivative

**Goal.** Compute the local ReLU slope, because backpropagation multiplies gradients by this derivative. We build it in 2 steps.

In [ ]:
z_b3 = np.array([-2., -0.5, 0., 0.5, 2.]) # choose points around the ReLU kink.
grad_b3 = (z_b3 > 0).astype(float) # derivative convention: 0 at and below zero, 1 above zero.
print("z:", z_b3) # inspect the pre-activations.
print("ReLU gradient:", grad_b3) # inspect which points pass gradient.

▶ What you'll see: only positive pre-activations get derivative 1.

In [ ]:
plt.figure(figsize=(4, 3)) # create a compact derivative chart.
plt.step(z_b3, grad_b3, where="post", color="purple") # show the piecewise derivative.
plt.ylim(-0.1, 1.1) # keep the derivative scale readable.
plt.title("Basic 3: ReLU derivative") # title the plot.
plt.show() # display the step chart.

▶ What you'll see: a jump from 0 to 1 at the activation threshold.

👀 Takeaway: inactive ReLUs block local gradient, while active ReLUs pass it unchanged.

### Basic 4 — Compute sigmoid probabilities

**Goal.** Map scalar scores into $(0,1)$ with sigmoid, because bounded outputs can represent probabilities or gates. We build it in 2 steps.

In [ ]:
z_b4 = np.array([-2., 0., 2.]) # choose low, central, and high scores.
sig_b4 = 1 / (1 + np.exp(-z_b4)) # compute sigmoid from its formula.
print("sigmoid values:", np.round(sig_b4, 3)) # inspect the bounded outputs.
assert round(float(sig_b4[1]), 3) == 0.500 # sigmoid(0)=0.5.

▶ What you'll see: negative scores map below 0.5, zero maps to 0.5, and positive scores map above 0.5.

In [ ]:
z_curve_b4 = np.linspace(-6, 6, 200) # create a smooth x-axis.
sig_curve_b4 = 1 / (1 + np.exp(-z_curve_b4)) # compute sigmoid curve.
plt.figure(figsize=(4, 3)) # create a curve plot.
plt.plot(z_curve_b4, sig_curve_b4, color="navy") # draw sigmoid.
plt.axhline(0.5, color="gray", linestyle="--") # mark the central probability.
plt.title("Basic 4: sigmoid squashes to (0,1)") # title the plot.
plt.show() # display the curve.

▶ What you'll see: sigmoid is S-shaped and bounded between 0 and 1.

👀 Takeaway: sigmoid turns a single logit into a smooth probability-like gate.

### Basic 5 — Inspect sigmoid saturation

**Goal.** Compute sigmoid derivatives in the tails, because saturation is a gradient-flow problem. We build it in 2 steps.

In [ ]:
z_b5 = np.array([-6., 0., 6.]) # choose saturated and central inputs.
s_b5 = 1 / (1 + np.exp(-z_b5)) # compute sigmoid values.
g_b5 = s_b5 * (1 - s_b5) # compute sigmoid derivative.
print("sigmoid:", np.round(s_b5, 4)) # inspect outputs.
print("derivative:", np.round(g_b5, 4)) # inspect local slopes.
assert round(float(g_b5[1]), 3) == 0.250 # maximum sigmoid slope occurs at zero.

▶ What you'll see: the derivative is near zero at ±6 but 0.25 at zero.

In [ ]:
plt.figure(figsize=(4, 3)) # create a derivative comparison plot.
plt.bar(["z=-6", "z=0", "z=6"], g_b5, color="crimson") # show saturation slopes.
plt.title("Basic 5: sigmoid derivative shrinks in tails") # title the plot.
plt.ylabel("local derivative") # label the gradient scale.
plt.show() # display the bars.

▶ What you'll see: the middle bar is much taller than the saturated tail bars.

👀 Takeaway: saturated sigmoid units make earlier-layer gradients very small.

### Basic 6 — Compare tanh to sigmoid

**Goal.** Center bounded activations with tanh, because zero-centered signals often optimize more cleanly than all-positive ones. We build it in 2 steps.

In [ ]:
z_b6 = np.array([-2., 0., 2.]) # choose symmetric inputs.
tanh_b6 = np.tanh(z_b6) # compute tanh values.
sig_b6 = 1 / (1 + np.exp(-z_b6)) # compute sigmoid values for comparison.
print("tanh:", np.round(tanh_b6, 3)) # inspect centered outputs.
print("sigmoid:", np.round(sig_b6, 3)) # inspect positive-only outputs.
assert round(float(tanh_b6[1]), 3) == 0.000 # tanh(0)=0.

▶ What you'll see: tanh is symmetric around zero, while sigmoid is centered at 0.5.

In [ ]:
z_curve_b6 = np.linspace(-4, 4, 200) # create curve inputs.
plt.figure(figsize=(4, 3)) # create comparison plot.
plt.plot(z_curve_b6, np.tanh(z_curve_b6), label="tanh") # draw tanh.
plt.plot(z_curve_b6, 1 / (1 + np.exp(-z_curve_b6)), label="sigmoid") # draw sigmoid.
plt.axhline(0, color="black", linewidth=0.6) # mark zero.
plt.title("Basic 6: tanh is zero-centered") # title the plot.
plt.legend() # show labels.
plt.show() # display the curves.

▶ What you'll see: tanh spans negative to positive values, while sigmoid stays positive.

👀 Takeaway: tanh is a bounded activation with centered outputs and saturated tails.

### Basic 7 — Build softmax for two logits

**Goal.** Convert two competing scores into probabilities, because classification usually needs relative preference rather than raw magnitude. We build it in 2 steps.

In [ ]:
logits_b7 = np.array([2.9, 0.4]) # define the lesson score and baseline.
exp_b7 = np.exp(logits_b7) # exponentiate both logits.
print("exp values:", np.round(exp_b7, 3)) # inspect the odds-like numbers.
assert round(float(exp_b7[0]), 3) == 18.174 # verify the lesson exponential.

▶ What you'll see: the larger logit becomes a much larger exponential.

In [ ]:
prob_b7 = exp_b7 / exp_b7.sum() # normalize exponentials into probabilities.
print("softmax probabilities:", np.round(prob_b7, 3)) # inspect the probability vector.
assert round(float(prob_b7[0]), 3) == 0.924 # verify the lesson probability.
plt.figure(figsize=(4, 3)) # create a probability bar chart.
plt.bar(["score", "baseline"], prob_b7, color=["teal", "gray"]) # plot probabilities.
plt.ylim(0, 1) # probability scale.
plt.title("Basic 7: two-class softmax") # title the plot.
plt.show() # display the bars.

▶ What you'll see: the first logit gets about 92.4% probability.

👀 Takeaway: softmax turns score gaps into normalized class probabilities.

### Basic 8 — Stabilize softmax by shifting logits

**Goal.** Subtract the maximum logit before exponentiating, because this prevents overflow without changing probabilities. We build it in 2 steps.

In [ ]:
logits_b8 = np.array([1002.9, 1000.4]) # define large logits with the same 2.5 gap.
shifted_b8 = logits_b8 - np.max(logits_b8) # subtract the largest logit for stability.
print("shifted logits:", shifted_b8) # inspect safe values before exponentiation.

▶ What you'll see: huge logits become `[0, -2.5]`, which are safe to exponentiate.

In [ ]:
prob_b8 = np.exp(shifted_b8) / np.exp(shifted_b8).sum() # compute stable softmax.
print("stable probabilities:", np.round(prob_b8, 3)) # inspect unchanged probabilities.
assert round(float(prob_b8[0]), 3) == 0.924 # same probability as Basic 7.
plt.figure(figsize=(4, 3)) # create a stable-softmax plot.
plt.bar(["class 0", "class 1"], prob_b8, color="darkorange") # show probabilities.
plt.ylim(0, 1) # probability scale.
plt.title("Basic 8: shifted softmax") # title the plot.
plt.show() # display the bars.

▶ What you'll see: the probabilities match the small-logit case despite the huge offset.

👀 Takeaway: softmax is invariant to adding or subtracting the same constant from every logit.

### Basic 9 — Normalize one activation value

**Goal.** Standardize a scalar activation, because scale changes the effective signal passed to later layers. We build it in 2 steps.

In [ ]:
z_b9 = 2.9 # define the raw activation value.
mean_b9 = 1.0 # define the reference mean.
var_b9 = 0.25 # define the reference variance.
eps_b9 = 1e-5 # define a small numerical stabilizer.
print("z, mean, variance:", z_b9, mean_b9, var_b9) # inspect normalization ingredients.

▶ What you'll see: the raw value is above the reference mean.

In [ ]:
norm_b9 = (z_b9 - mean_b9) / np.sqrt(var_b9 + eps_b9) # normalize by standard deviation.
print("normalized value:", round(norm_b9, 3)) # inspect standardized scale.
assert round(norm_b9, 3) == 3.800 # verify the lesson normalization number.
plt.figure(figsize=(4, 3)) # create a small comparison chart.
plt.bar(["raw z", "normalized"], [z_b9, norm_b9], color=["gray", "seagreen"]) # compare scales.
plt.title("Basic 9: activation normalization") # title the plot.
plt.show() # display the bars.

▶ What you'll see: the normalized value says the activation is unusually high relative to the reference scale.

👀 Takeaway: normalization changes values into standardized units that affect gradient and downstream behavior.

### Basic 10 — Count saved activation memory

**Goal.** Compute a tiny activation-memory cost, because backpropagation must store activations for later gradient calculations. We build it in 2 steps.

In [ ]:
vectors_b10 = 5 # define how many activation vectors are saved.
width_b10 = 128 # define vector length.
bytes_b10 = 4 # define bytes per float32 number.
print("vectors:", vectors_b10, "width:", width_b10, "bytes/float:", bytes_b10) # inspect memory ingredients.

▶ What you'll see: the example stores 5 vectors of length 128 in 32-bit floats.

In [ ]:
memory_kb_b10 = vectors_b10 * width_b10 * bytes_b10 / 1024 # convert bytes to KB.
print("memory KB:", round(memory_kb_b10, 3)) # inspect the storage cost.
assert round(memory_kb_b10, 3) == 2.500 # verify the lesson memory number.
plt.figure(figsize=(4, 3)) # create a memory chart.
plt.bar(["saved activations"], [memory_kb_b10], color="purple") # show memory cost.
plt.ylabel("KB") # label units.
plt.title("Basic 10: activation storage") # title the plot.
plt.show() # display the bar.

▶ What you'll see: even this tiny block stores 2.5 KB, and real networks multiply this by layers and batches.

👀 Takeaway: activation design has hardware consequences because saved forward values are needed by backpropagation.

## 🟡 Easy

### Easy 1 — Compare activation curves on one axis

**Goal.** Plot sigmoid, tanh, ReLU, and GELU together, because activation choice is easiest to understand by comparing shape and scale. We build it in 3 steps.

In [ ]:
z_e1 = np.linspace(-4, 4, 300) # create a shared input axis for all activations.
sig_e1 = 1 / (1 + np.exp(-z_e1)) # compute sigmoid values.
tanh_e1 = np.tanh(z_e1) # compute tanh values.
relu_e1 = np.maximum(0, z_e1) # compute ReLU values.
print("grid size:", z_e1.size) # inspect how many points are plotted.

▶ What you'll see: all activations are evaluated on the same 300-point grid.

In [ ]:
gelu_e1 = 0.5 * z_e1 * (1 + np.tanh(np.sqrt(2 / np.pi) * (z_e1 + 0.044715 * z_e1 ** 3))) # compute approximate GELU.
print("values at z=0 approx:", round(float(sig_e1[150]), 3), round(float(tanh_e1[150]), 3), round(float(relu_e1[150]), 3), round(float(gelu_e1[150]), 3)) # inspect center behavior.

In [ ]:
plt.figure(figsize=(5, 3)) # create comparison plot.
plt.plot(z_e1, sig_e1, label="sigmoid") # draw sigmoid.
plt.plot(z_e1, tanh_e1, label="tanh") # draw tanh.
plt.plot(z_e1, relu_e1, label="ReLU") # draw ReLU.
plt.plot(z_e1, gelu_e1, label="GELU") # draw GELU.
plt.axhline(0, color="black", linewidth=0.6) # mark zero.
plt.title("Easy 1: activation shapes") # title the plot.
plt.legend() # show labels.
plt.show() # display curves.

▶ What you'll see: bounded activations flatten, ReLU grows linearly, and GELU smoothly gates around zero.

👀 Takeaway: the forward curve determines both signal range and the kind of nonlinearity the network can compose.

### Easy 2 — Backpropagate through different activations

**Goal.** Multiply one upstream gradient by local derivatives, because activation derivatives decide how much error reaches earlier weights. We build it in 3 steps.

In [ ]:
z_e2 = np.array([-4., 0., 2.9]) # choose saturated, central, and active-positive locations.
up_e2 = 1.65 # define one upstream gradient from later layers.
sig_e2 = 1 / (1 + np.exp(-z_e2)) # compute sigmoid outputs.
print("upstream gradient:", up_e2) # inspect the incoming gradient.

▶ What you'll see: the same upstream gradient will be filtered three different ways.

In [ ]:
grad_sig_e2 = sig_e2 * (1 - sig_e2) # sigmoid derivative.
grad_tanh_e2 = 1 - np.tanh(z_e2) ** 2 # tanh derivative.
grad_relu_e2 = (z_e2 > 0).astype(float) # ReLU derivative.
print("local derivatives sigmoid/tanh/ReLU:") # label the table.
print(np.round(np.vstack([grad_sig_e2, grad_tanh_e2, grad_relu_e2]), 4)) # inspect local slopes.

In [ ]:
back_e2 = np.vstack([up_e2 * grad_sig_e2, up_e2 * grad_tanh_e2, up_e2 * grad_relu_e2]) # apply chain rule.
print("backpropagated gradients:", np.round(back_e2, 4)) # inspect scaled gradients.
assert round(float(back_e2[2, 2]), 3) == 1.650 # active ReLU passes the gradient.
plt.figure(figsize=(5, 3)) # create a grouped heatmap-like plot.
plt.imshow(back_e2, cmap="viridis", aspect="auto") # visualize gradient magnitudes.
plt.colorbar(label="gradient") # add scale.
plt.yticks([0, 1, 2], ["sigmoid", "tanh", "ReLU"]) # label rows.
plt.xticks([0, 1, 2], ["z=-4", "z=0", "z=2.9"]) # label columns.
plt.title("Easy 2: activation gradients") # title plot.
plt.show() # display heatmap.

▶ What you'll see: saturated nonlinearities shrink gradients, while active ReLU preserves the upstream value.

👀 Takeaway: activation choice changes optimization because the chain rule multiplies by local slopes.

### Easy 3 — Train one scalar through a ReLU gate

**Goal.** Take repeated gradient steps on one weight, because the lesson's scalar update becomes learning only when repeated. We build it in 3 steps.

In [ ]:
x_e3 = 1.0 # use one input feature.
w_e3 = 2.0 # initialize one scalar weight.
target_e3 = 0.5 # set a small desired activation.
eta_e3 = 0.08 # use the lesson learning rate.
history_e3 = [] # store weight and loss values.
print("initial weight:", w_e3) # inspect starting point.

▶ What you'll see: the scalar parameter starts at 2.0.

In [ ]:
for step_e3 in range(20): # perform repeated reliable nudges.
    z_step_e3 = w_e3 * x_e3 # affine pre-activation.
    h_step_e3 = np.maximum(0, z_step_e3) # ReLU output.
    loss_step_e3 = 0.5 * (h_step_e3 - target_e3) ** 2 # squared error with convenient 1/2.
    grad_h_e3 = h_step_e3 - target_e3 # derivative of loss wrt activation.
    grad_w_e3 = grad_h_e3 * (z_step_e3 > 0) * x_e3 # chain rule through ReLU and z=w*x.
    w_e3 = w_e3 - eta_e3 * grad_w_e3 # gradient descent update.
    history_e3.append([w_e3, loss_step_e3]) # save progress.
print("final weight:", round(w_e3, 3), "final loss:", round(history_e3[-1][1], 4)) # inspect convergence.
assert history_e3[-1][1] < history_e3[0][1] # verify learning reduced loss.

In [ ]:
hist_e3 = np.array(history_e3) # convert history to an array for plotting.
plt.figure(figsize=(5, 3)) # create a learning-curve plot.
plt.plot(hist_e3[:, 1], color="teal") # plot loss over steps.
plt.title("Easy 3: repeated ReLU-gradient steps") # title the plot.
plt.xlabel("step") # label x-axis.
plt.ylabel("loss") # label y-axis.
plt.show() # display curve.

▶ What you'll see: loss decreases because the active ReLU passes gradient to the weight.

👀 Takeaway: one update is arithmetic; learning is the repeated application of that local gradient rule.

### Easy 4 — Detect dead ReLU units in a batch

**Goal.** Count inactive ReLUs across a minibatch, because units with all-negative pre-activations send no gradient through ReLU. We build it in 3 steps.

In [ ]:
Z_e4 = np.array([[-2.0, 0.5, -1.0], [-1.5, 1.2, -0.7], [-0.4, 0.8, -2.2], [-3.0, 0.1, -0.3]]) # four examples by three units.
H_e4 = np.maximum(0, Z_e4) # apply ReLU elementwise.
print("ReLU activations:\n", H_e4) # inspect which units are active.

▶ What you'll see: the first and third units are zero for every example, while the middle unit activates.

In [ ]:
active_counts_e4 = np.sum(H_e4 > 0, axis=0) # count positive outputs per unit.
dead_e4 = active_counts_e4 == 0 # mark units that never activated in the batch.
print("active counts:", active_counts_e4) # inspect activity per unit.
print("dead units:", dead_e4.astype(int)) # inspect dead-unit flags.
assert int(dead_e4.sum()) == 2 # two units are dead on this batch.

In [ ]:
plt.figure(figsize=(4, 3)) # create an activity chart.
plt.bar(["unit0", "unit1", "unit2"], active_counts_e4, color=["crimson" if d else "seagreen" for d in dead_e4]) # color dead units red.
plt.title("Easy 4: ReLU activity counts") # title plot.
plt.ylabel("active examples") # label y-axis.
plt.show() # display bars.

▶ What you'll see: units with zero active examples are obvious red bars at height 0.

👀 Takeaway: ReLU sparsity is useful, but all-zero activity means no local gradient for that batch.

### Easy 5 — Compute cross-entropy from softmax

**Goal.** Convert logits to a probability and loss, because softmax usually drives learning through cross-entropy. We build it in 3 steps.

In [ ]:
logits_e5 = np.array([2.9, 0.4, -0.2]) # define three class scores.
y_e5 = 0 # choose class 0 as the true label.
shift_e5 = logits_e5 - np.max(logits_e5) # stabilize logits before exponentials.
print("shifted logits:", shift_e5) # inspect stable inputs.

▶ What you'll see: the largest logit is shifted to zero and the others become negative gaps.

In [ ]:
probs_e5 = np.exp(shift_e5) / np.exp(shift_e5).sum() # compute stable softmax probabilities.
loss_e5 = -np.log(probs_e5[y_e5]) # compute cross-entropy for the true class.
print("probabilities:", np.round(probs_e5, 3)) # inspect class probabilities.
print("cross-entropy:", round(float(loss_e5), 3)) # inspect negative log likelihood.
assert abs(float(probs_e5.sum()) - 1.0) < 1e-12 # probabilities must sum to one.

In [ ]:
plt.figure(figsize=(4, 3)) # create probability chart.
plt.bar(["c0", "c1", "c2"], probs_e5, color=["seagreen", "gray", "gray"]) # highlight true class.
plt.title(f"Easy 5: softmax loss = {loss_e5:.3f}") # title plot with loss.
plt.ylim(0, 1) # probability scale.
plt.show() # display bars.

▶ What you'll see: the true class receives the highest probability, so the cross-entropy is modest.

👀 Takeaway: softmax plus cross-entropy rewards putting high probability on the correct class.

## 🔴 Advanced

### Advanced 1 — Track vanishing gradients through many layers

**Goal.** Multiply activation derivatives across depth, because deep networks can lose gradient when many local slopes are below one. We build it in 4 steps.

In [ ]:
depths_a1 = np.arange(1, 21) # evaluate chain depths from 1 to 20.
sig_slope_a1 = 0.25 # largest possible sigmoid derivative.
tanh_slope_a1 = 0.50 # illustrative moderate tanh derivative.
relu_slope_a1 = 1.00 # active ReLU derivative.
print("depths:", depths_a1[:5], "...") # inspect the sweep.

▶ What you'll see: the experiment checks progressively deeper chains.

In [ ]:
grad_sig_a1 = sig_slope_a1 ** depths_a1 # repeated sigmoid slopes shrink fast.
grad_tanh_a1 = tanh_slope_a1 ** depths_a1 # repeated moderate slopes also shrink.
grad_relu_a1 = relu_slope_a1 ** depths_a1 # active ReLU preserves magnitude.
print("depth 10 gradients:", round(float(grad_sig_a1[9]), 8), round(float(grad_tanh_a1[9]), 8), round(float(grad_relu_a1[9]), 3)) # inspect depth-10 values.
assert round(float(grad_sig_a1[1]), 4) == 0.0625 # 0.25^2.

In [ ]:
plt.figure(figsize=(5, 3)) # create gradient decay plot.
plt.plot(depths_a1, grad_sig_a1, marker="o", label="sigmoid max slope 0.25") # plot sigmoid chain.
plt.plot(depths_a1, grad_tanh_a1, marker="s", label="slope 0.5") # plot moderate chain.
plt.plot(depths_a1, grad_relu_a1, label="active ReLU slope 1") # plot active ReLU chain.
plt.yscale("log") # use log scale for tiny gradients.
plt.title("Advanced 1: products of local slopes") # title plot.
plt.xlabel("number of layers") # label depth axis.
plt.ylabel("gradient multiplier") # label multiplier axis.
plt.legend() # show labels.
plt.show() # display curves.

▶ What you'll see: products of slopes below one collapse exponentially with depth.

In [ ]:
ratio_a1 = grad_relu_a1[-1] / grad_sig_a1[-1] # compare depth-20 active ReLU to sigmoid best-case.
print("ReLU/sigmoid multiplier at depth 20:", f"{ratio_a1:.2e}") # inspect the scale gap.

▶ What you'll see: a huge ratio, showing why local activation slopes matter globally.

👀 Takeaway: vanishing gradients are products of many local derivatives, not a mysterious separate phenomenon.

### Advanced 2 — Compare ReLU and leaky ReLU for dead units

**Goal.** Give negative pre-activations a small slope, because leaky ReLU keeps some gradient alive where ReLU would block it. We build it in 4 steps.

In [ ]:
z_a2 = np.linspace(-3, 3, 121) # create a shared input range.
alpha_a2 = 0.05 # choose a small negative-side slope.
relu_a2 = np.maximum(0, z_a2) # compute ReLU.
leaky_a2 = np.where(z_a2 > 0, z_a2, alpha_a2 * z_a2) # compute leaky ReLU.
print("alpha:", alpha_a2) # inspect the leak strength.

▶ What you'll see: the leaky version uses 5% slope for negative inputs.

In [ ]:
grad_relu_a2 = (z_a2 > 0).astype(float) # ReLU derivative.
grad_leaky_a2 = np.where(z_a2 > 0, 1.0, alpha_a2) # leaky derivative.
neg_idx_a2 = np.where(z_a2 < 0)[0][0] # choose one negative location.
print("negative gradients ReLU/leaky:", grad_relu_a2[neg_idx_a2], grad_leaky_a2[neg_idx_a2]) # inspect gradient rescue.
assert round(float(grad_leaky_a2[neg_idx_a2]), 2) == 0.05 # verify leak slope.

In [ ]:
plt.figure(figsize=(5, 3)) # create activation comparison.
plt.plot(z_a2, relu_a2, label="ReLU") # draw ReLU.
plt.plot(z_a2, leaky_a2, label="leaky ReLU") # draw leaky ReLU.
plt.axhline(0, color="black", linewidth=0.6) # mark zero.
plt.title("Advanced 2: leaky ReLU keeps a negative slope") # title plot.
plt.legend() # labels.
plt.show() # display curves.

▶ What you'll see: leaky ReLU is slightly negative on the left instead of flat zero.

In [ ]:
plt.figure(figsize=(5, 3)) # create derivative comparison.
plt.plot(z_a2, grad_relu_a2, label="ReLU derivative") # draw ReLU derivative.
plt.plot(z_a2, grad_leaky_a2, label="leaky derivative") # draw leaky derivative.
plt.ylim(-0.05, 1.1) # derivative scale.
plt.title("Advanced 2: derivative comparison") # title plot.
plt.legend() # show labels.
plt.show() # display derivatives.

▶ What you'll see: leaky ReLU has a nonzero left-side derivative, so negative units can still update.

👀 Takeaway: leaky ReLU trades exact sparsity for better gradient flow on the negative side.

### Advanced 3 — Estimate a GELU derivative numerically

**Goal.** Approximate the GELU slope with finite differences, because smooth gates can be understood by inspecting how output changes under tiny input moves. We build it in 4 steps.

In [ ]:
def gelu_a3(z):
    return 0.5 * z * (1 + np.tanh(np.sqrt(2 / np.pi) * (z + 0.044715 * z ** 3)))

z_a3 = np.linspace(-3, 3, 121) # create a smooth input grid.
h_a3 = gelu_a3(z_a3) # compute GELU outputs.
print("GELU(1):", round(float(gelu_a3(np.array([1.0]))[0]), 3)) # inspect a worked value.
assert round(float(gelu_a3(np.array([1.0]))[0]), 3) == 0.841 # verify common GELU value.

▶ What you'll see: GELU(1) is about 0.841, less than identity but much more than zero.

In [ ]:
eps_a3 = 1e-4 # choose a tiny finite-difference step.
grad_num_a3 = (gelu_a3(z_a3 + eps_a3) - gelu_a3(z_a3 - eps_a3)) / (2 * eps_a3) # central difference derivative.
print("gradient near zero:", round(float(grad_num_a3[np.argmin(np.abs(z_a3))]), 3)) # inspect central slope.

In [ ]:
plt.figure(figsize=(5, 3)) # create output and derivative plot.
plt.plot(z_a3, h_a3, label="GELU") # draw activation.
plt.plot(z_a3, grad_num_a3, label="numeric derivative") # draw derivative.
plt.axhline(0, color="black", linewidth=0.6) # mark zero.
plt.title("Advanced 3: GELU and its slope") # title plot.
plt.legend() # labels.
plt.show() # display plot.

▶ What you'll see: the derivative changes smoothly rather than jumping like ReLU.

In [ ]:
min_grad_a3 = float(np.min(grad_num_a3)) # inspect the smallest numerical slope on the grid.
max_grad_a3 = float(np.max(grad_num_a3)) # inspect the largest numerical slope.
print("GELU derivative range:", round(min_grad_a3, 3), "to", round(max_grad_a3, 3)) # summarize slopes.

▶ What you'll see: GELU's slope can be small or even slightly negative on the left, then exceeds 1 around the transition.

👀 Takeaway: smooth activations can have richer gradient behavior than piecewise-linear gates.

### Advanced 4 — Compute the softmax Jacobian

**Goal.** Build the matrix $\partial p_i/\partial z_j$, because softmax probabilities are coupled: changing one logit changes every probability. We build it in 4 steps.

In [ ]:
logits_a4 = np.array([2.9, 0.4, -0.2]) # define three competing logits.
shift_a4 = logits_a4 - np.max(logits_a4) # stabilize before exponentials.
p_a4 = np.exp(shift_a4) / np.exp(shift_a4).sum() # compute softmax probabilities.
print("probabilities:", np.round(p_a4, 3)) # inspect p.
assert abs(float(p_a4.sum()) - 1.0) < 1e-12 # verify normalization.

▶ What you'll see: the probabilities are positive and sum to one.

In [ ]:
J_a4 = np.diag(p_a4) - np.outer(p_a4, p_a4) # softmax Jacobian formula.
print("Jacobian:\n", np.round(J_a4, 4)) # inspect probability coupling.
print("row sums:", np.round(J_a4.sum(axis=1), 8)) # each row sums to zero due to shift invariance.
assert np.allclose(J_a4.sum(axis=1), 0.0) # changing all logits equally changes no probabilities.

In [ ]:
plt.figure(figsize=(4, 3)) # create a Jacobian heatmap.
plt.imshow(J_a4, cmap="coolwarm", aspect="auto") # visualize positive diagonal and negative off-diagonal terms.
plt.colorbar(label="derivative") # add scale.
plt.title("Advanced 4: softmax Jacobian") # title plot.
plt.xlabel("logit j") # label columns.
plt.ylabel("probability i") # label rows.
plt.show() # display heatmap.

▶ What you'll see: diagonal terms are positive and off-diagonal terms are negative because classes compete.

In [ ]:
g_a4 = np.array([0.2, -0.1, -0.1]) # define an upstream gradient wrt probabilities.
back_a4 = J_a4 @ g_a4 # backpropagate through softmax.
print("gradient wrt logits:", np.round(back_a4, 4)) # inspect coupled logit gradients.
assert abs(float(back_a4.sum())) < 1e-12 # logit gradients sum to zero under softmax coupling.

▶ What you'll see: every logit gradient depends on all probability gradients, not just its own class.

👀 Takeaway: softmax is a coupled activation, so its derivative is a full matrix rather than independent scalar slopes.

### Advanced 5 — Simulate activation variance across layers

**Goal.** Propagate random data through several layers, because activation choice changes signal scale before any learning happens. We build it in 4 steps.

In [ ]:
rng_a5 = np.random.default_rng(0) # create reproducible random numbers.
X_a5 = rng_a5.normal(size=(512, 64)) # create a minibatch of 512 examples with 64 features.
layers_a5 = 8 # choose a small depth for the simulation.
print("input variance:", round(float(np.var(X_a5)), 3)) # inspect starting scale.

▶ What you'll see: the input variance is close to 1.

In [ ]:
vars_relu_a5 = [] # store variance after each ReLU layer.
vars_tanh_a5 = [] # store variance after each tanh layer.
H_relu_a5 = X_a5.copy() # initialize ReLU path.
H_tanh_a5 = X_a5.copy() # initialize tanh path.
for layer_a5 in range(layers_a5): # propagate through random layers.
    W_a5 = rng_a5.normal(scale=np.sqrt(2 / H_relu_a5.shape[1]), size=(H_relu_a5.shape[1], 64)) # He-scaled weights for ReLU path.
    H_relu_a5 = np.maximum(0, H_relu_a5 @ W_a5) # affine transform then ReLU.
    H_tanh_a5 = np.tanh(H_tanh_a5 @ W_a5) # same weights then tanh.
    vars_relu_a5.append(float(np.var(H_relu_a5))) # record ReLU variance.
    vars_tanh_a5.append(float(np.var(H_tanh_a5))) # record tanh variance.
print("final variances ReLU/tanh:", round(vars_relu_a5[-1], 3), round(vars_tanh_a5[-1], 3)) # inspect final scales.

In [ ]:
plt.figure(figsize=(5, 3)) # create scale-tracking plot.
plt.plot(np.arange(1, layers_a5 + 1), vars_relu_a5, marker="o", label="ReLU") # plot ReLU variance.
plt.plot(np.arange(1, layers_a5 + 1), vars_tanh_a5, marker="s", label="tanh") # plot tanh variance.
plt.title("Advanced 5: activation variance across depth") # title plot.
plt.xlabel("layer") # label x-axis.
plt.ylabel("activation variance") # label y-axis.
plt.legend() # show labels.
plt.show() # display curves.

▶ What you'll see: ReLU and tanh maintain different variance patterns because one gates and one saturates.

In [ ]:
memory_mb_a5 = layers_a5 * H_relu_a5.shape[0] * H_relu_a5.shape[1] * 4 / (1024 ** 2) # saved activation memory for one path.
print("saved activation memory MB:", round(memory_mb_a5, 3)) # inspect hardware cost.
assert memory_mb_a5 > 0 # basic sanity check for memory bookkeeping.

▶ What you'll see: even this toy stack needs saved activation storage, and real batches/layers scale the number up.

👀 Takeaway: activation functions shape both numerical scale and the memory footprint that training must carry.

---

# Reference walkthrough — original compact notebook

The sections above build every idea from scratch with detailed steps and worked examples. Below is the original compact notebook for this lesson, kept as a concise reference and for its practice prompts.

Activations decide which signals pass, which saturate, and which become probabilities.

The affine score is not enough; the nonlinearity controls signal flow. We verify sigmoid, tanh, ReLU, GELU, and softmax before sweeping activations on the same ladder. Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits
from sklearn.datasets import make_blobs
from sklearn.datasets import make_moons
from sklearn.metrics import accuracy_score
from sklearn.metrics import log_loss
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(6)


def clf_digits_ladder():
    """D1 XOR -> D2 blobs -> D3 noisy moons -> D4 digits -> D5 noisy digits."""
    rungs = []

    x1 = np.array([[0.0, 0.0], [1.0, 1.0], [0.0, 1.0], [1.0, 0.0]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 XOR", x1, y1))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=1.0, random_state=1)
    rungs.append(("D2 blobs (3-class)", x2, y2))

    x3, y3 = make_moons(n_samples=300, noise=0.3, random_state=2)
    rungs.append(("D3 noisy moons", x3, y3))

    digits = load_digits()
    xd = digits.data / 16.0
    rungs.append(("D4 digits (real, 10-class, 64-D)", xd, digits.target))

    rng = np.random.default_rng(5)
    xn = xd + rng.normal(0.0, 0.25, size=xd.shape)
    yn = digits.target.copy()
    flip = rng.random(yn.shape) < 0.1
    yn[flip] = rng.integers(0, 10, size=int(flip.sum()))
    rungs.append(("D5 digits + label/feature noise", xn, yn))

    return rungs


def split_scale(X, y):
    if len(y) < 20:
        scaler = StandardScaler()
        x_scaled = scaler.fit_transform(X)
        return x_scaled, x_scaled, y.copy(), y.copy(), scaler

    x_train, x_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.4,
        random_state=0,
        stratify=y,
    )
    rng = np.random.default_rng(606)
    if len(y_train) > 600:
        train_idx = rng.choice(len(y_train), size=600, replace=False)
        y_train = y_train[train_idx]
        x_train = x_train[train_idx]
    if len(y_test) > 300:
        test_idx = rng.choice(len(y_test), size=300, replace=False)
        y_test = y_test[test_idx]
        x_test = x_test[test_idx]

    scaler = StandardScaler()
    x_train = scaler.fit_transform(x_train)
    x_test = scaler.transform(x_test)
    return x_train, x_test, y_train, y_test, scaler


def one_hot(y, classes):
    out = np.zeros((len(y), classes))
    out[np.arange(len(y)), y.astype(int)] = 1.0
    return out


def stable_softmax(logits):
    shifted = logits - logits.max(axis=1, keepdims=True)
    exp_values = np.exp(shifted)
    return exp_values / exp_values.sum(axis=1, keepdims=True)


def activation_forward(z, name):
    if name == "relu":
        return np.maximum(0.0, z)
    if name == "tanh":
        return np.tanh(z)
    if name == "sigmoid":
        return 1.0 / (1.0 + np.exp(-np.clip(z, -40.0, 40.0)))
    if name == "gelu":
        c = math.sqrt(2.0 / math.pi)
        return 0.5 * z * (1.0 + np.tanh(c * (z + 0.044715 * z**3)))
    return z


def activation_backward(z, name):
    if name == "relu":
        return (z > 0.0).astype(float)
    if name == "tanh":
        h = np.tanh(z)
        return 1.0 - h**2
    if name == "sigmoid":
        h = 1.0 / (1.0 + np.exp(-np.clip(z, -40.0, 40.0)))
        return h * (1.0 - h)
    if name == "gelu":
        c = math.sqrt(2.0 / math.pi)
        u = c * (z + 0.044715 * z**3)
        t = np.tanh(u)
        sech2 = 1.0 - t**2
        return 0.5 * (1.0 + t) + 0.5 * z * sech2 * c * (1.0 + 3.0 * 0.044715 * z**2)
    return np.ones_like(z)


def initialize_mlp(n_features, n_hidden, n_classes, seed):
    rng = np.random.default_rng(seed)
    w1 = rng.normal(0.0, math.sqrt(2.0 / max(1, n_features)), size=(n_features, n_hidden))
    b1 = np.zeros(n_hidden)
    w2 = rng.normal(0.0, math.sqrt(2.0 / max(1, n_hidden)), size=(n_hidden, n_classes))
    b2 = np.zeros(n_classes)
    return {"w1": w1, "b1": b1, "w2": w2, "b2": b2}


def forward(params, X, activation):
    z1 = X @ params["w1"] + params["b1"]
    h1 = activation_forward(z1, activation)
    logits = h1 @ params["w2"] + params["b2"]
    probs = stable_softmax(logits)
    cache = {"z1": z1, "h1": h1, "logits": logits, "probs": probs}
    return probs, cache


def compute_loss(probs, y, loss_name):
    classes = probs.shape[1]
    targets = one_hot(y, classes)
    eps = 1e-9

    if loss_name == "mse":
        return float(np.mean((probs - targets) ** 2))

    if loss_name == "hinge":
        correct = probs[np.arange(len(y)), y]
        margins = np.maximum(0.0, probs - correct[:, None] + 0.2)
        margins[np.arange(len(y)), y] = 0.0
        return float(np.mean(np.sum(margins, axis=1)))

    return float(-np.mean(np.log(probs[np.arange(len(y)), y] + eps)))


def output_gradient(probs, y, loss_name):
    classes = probs.shape[1]
    targets = one_hot(y, classes)
    n = max(1, len(y))

    if loss_name == "mse":
        return 2.0 * (probs - targets) / (n * classes)

    if loss_name == "hinge":
        correct = probs[np.arange(len(y)), y]
        active = probs - correct[:, None] + 0.2 > 0.0
        active[np.arange(len(y)), y] = False
        grad = active.astype(float)
        grad[np.arange(len(y)), y] = -grad.sum(axis=1)
        return grad / n

    return (probs - targets) / n


def train_tiny_mlp(
    X,
    y,
    hidden=24,
    epochs=25,
    lr=0.12,
    activation="relu",
    loss_name="ce",
    seed=0,
):
    x_train, x_test, y_train, y_test, scaler = split_scale(X, y)
    classes = int(np.max(y)) + 1
    params = initialize_mlp(x_train.shape[1], hidden, classes, seed)
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    for epoch in range(epochs):
        probs, cache = forward(params, x_train, activation)
        grad_logits = output_gradient(probs, y_train, loss_name)
        grad_w2 = cache["h1"].T @ grad_logits
        grad_b2 = grad_logits.sum(axis=0)
        grad_h = grad_logits @ params["w2"].T
        grad_z1 = grad_h * activation_backward(cache["z1"], activation)
        grad_w1 = x_train.T @ grad_z1
        grad_b1 = grad_z1.sum(axis=0)

        params["w1"] = params["w1"] - lr * grad_w1
        params["b1"] = params["b1"] - lr * grad_b1
        params["w2"] = params["w2"] - lr * grad_w2
        params["b2"] = params["b2"] - lr * grad_b2

        train_probs, _ = forward(params, x_train, activation)
        val_probs, _ = forward(params, x_test, activation)
        train_pred = train_probs.argmax(axis=1)
        val_pred = val_probs.argmax(axis=1)
        history["train_loss"].append(compute_loss(train_probs, y_train, loss_name))
        history["val_loss"].append(compute_loss(val_probs, y_test, loss_name))
        history["train_acc"].append(float(accuracy_score(y_train, train_pred)))
        history["val_acc"].append(float(accuracy_score(y_test, val_pred)))

    result = {
        "params": params,
        "history": history,
        "scaler": scaler,
        "x_test": x_test,
        "y_test": y_test,
        "activation": activation,
        "loss_name": loss_name,
    }
    return result


def predict_tiny(model, X_raw):
    X = model["scaler"].transform(X_raw)
    probs, _ = forward(model["params"], X, model["activation"])
    return probs.argmax(axis=1)


def evaluate_ladder(hidden=24, epochs=25, lr=0.12, activation="relu", loss_name="ce", seed=0):
    rows = []
    models = []
    for idx, (name, X, y) in enumerate(clf_digits_ladder(), start=1):
        model = train_tiny_mlp(
            X,
            y,
            hidden=hidden,
            epochs=epochs,
            lr=lr,
            activation=activation,
            loss_name=loss_name,
            seed=seed + idx,
        )
        metric = model["history"]["val_acc"][-1]
        loss_value = model["history"]["val_loss"][-1]
        rows.append({"rung": idx, "name": name, "accuracy": metric, "loss": loss_value})
        models.append(model)
    return rows, models


def print_rows(rows, metric="accuracy"):
    print("rung | dataset | accuracy | loss")
    for row in rows:
        print(f"D{row['rung']} | {row['name']} | {row['accuracy']:.3f} | {row['loss']:.3f}")


def plot_decision_panel(ax, model, X, y, title):
    if X.shape[1] != 2:
        side = int(math.sqrt(X.shape[1]))
        if side * side == X.shape[1]:
            ax.imshow(X[0].reshape(side, side), cmap="gray")
            pred = predict_tiny(model, X[:1])[0]
            ax.set_title(f"{title}\ny={int(y[0])}, pred={int(pred)}")
        else:
            ax.plot(X[0])
            ax.set_title(title)
        ax.set_xticks([])
        ax.set_yticks([])
        return

    x_min = X[:, 0].min() - 0.8
    x_max = X[:, 0].max() + 0.8
    y_min = X[:, 1].min() - 0.8
    y_max = X[:, 1].max() + 0.8
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 40), np.linspace(y_min, y_max, 40))
    grid = np.c_[xx.ravel(), yy.ravel()]
    zz = predict_tiny(model, grid).reshape(xx.shape)
    ax.contourf(xx, yy, zz, alpha=0.25, levels=np.arange(int(np.max(y)) + 2) - 0.5)
    ax.scatter(X[:, 0], X[:, 1], c=y, s=16, edgecolor="k", linewidth=0.2)
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])


def closing_figure(rows, models, metric="accuracy"):
    rungs = clf_digits_ladder()
    fig, axes = plt.subplots(2, 5, figsize=(16, 6))
    for ax, model, rung in zip(axes[0], models, rungs):
        name, X, y = rung
        plot_decision_panel(ax, model, X, y, name.split(" (")[0])

    xs = [row["rung"] for row in rows]
    ys = [row[metric] for row in rows]
    axes[1, 0].plot(xs, ys, marker="o")
    axes[1, 0].set_xticks(xs)
    axes[1, 0].set_xlabel("ladder rung")
    axes[1, 0].set_ylabel(metric)
    axes[1, 0].set_title(f"{metric} vs ladder difficulty")
    for extra in axes[1, 1:]:
        extra.axis("off")
    fig.tight_layout()
    plt.show()


def plot_history(models, metric="val_acc"):
    plt.figure(figsize=(7, 4))
    for idx, model in enumerate(models, start=1):
        plt.plot(model["history"][metric], label=f"D{idx}")
    plt.xlabel("epoch")
    plt.ylabel(metric)
    plt.legend(ncol=3)
    plt.title(f"Training trace: {metric}")
    plt.show()


## Build the concept once on D1

The lesson formula is

$$h=\phi(z),\quad \mathrm{softmax}(z)_i=\frac{e^{z_i}}{\sum_j e^{z_j}}$$

We plug in the lesson's own numbers and assert the exact rounded values before using the method on the ladder.

In [ ]:

def activation_sweep(z):
    values = {
        "sigmoid": activation_forward(np.array([[z]]), "sigmoid")[0, 0],
        "tanh": activation_forward(np.array([[z]]), "tanh")[0, 0],
        "relu": activation_forward(np.array([[z]]), "relu")[0, 0],
        "gelu": activation_forward(np.array([[z]]), "gelu")[0, 0],
    }
    return values

x = np.array([1.5, -0.5])
z = float(np.array([1.6, 0.2]) @ x + 0.6)
h = max(0.0, z)
updated = 2.0 - 0.080 * 1.650
prob = math.exp(z) / (math.exp(z) + math.exp(0.4))
normalized = (z - 1.0) / math.sqrt(0.250 + 0.00001)
memory_kb = 5 * 128 * 4 / 1024
probs = stable_softmax(np.array([[z, 0.4, -0.2]]))[0]

assert round(z, 3) == 2.900
assert round(h, 3) == 2.900
assert round(updated, 3) == 1.868
assert round(math.exp(z), 3) == 18.174
assert round(prob, 3) == 0.924
assert round(normalized, 3) == 3.800
assert round(memory_kb, 3) == 2.500
assert round(float(probs.sum()), 6) == 1.000000

print("activation values at lesson z:", activation_sweep(z))
print("3-class softmax:", probs)


## Package the reusable method

The same tiny MLP code above performs a real forward pass, computes a real loss, backpropagates analytic gradients, and updates weights on CPU.

In [ ]:
rungs = clf_digits_ladder()
print("Reusable method: train_tiny_mlp + forward + analytic backward pass")
print("Rungs ready:", len(rungs))

## The dataset ladder

D1 is XOR, then blobs, noisy moons, real sklearn digits, and D5 digits with feature plus label noise. The method sees one feature matrix and one class vector at every rung.

In [ ]:
for idx, (name, X, y) in enumerate(clf_digits_ladder(), start=1):
    labels, counts = np.unique(y, return_counts=True)
    print(f"D{idx}: {name}")
    print("  X shape:", X.shape)
    print("  classes:", labels.tolist())
    print("  first counts:", counts[:5].tolist())
    print("  first row sample:", np.round(X[0, : min(8, X.shape[1])], 3))

## Run the same method across D1–D5

The tracked metric is accuracy.

In [ ]:
activations = ["sigmoid", "tanh", "relu", "gelu", "relu"]
rows = []
models = []
for activation, rung in zip(activations, clf_digits_ladder()):
    name, X, y = rung
    model = train_tiny_mlp(X, y, hidden=28, epochs=25, lr=0.12, activation=activation, loss_name="ce", seed=630 + len(rows))
    rows.append({"rung": len(rows) + 1, "name": name + " / " + activation, "accuracy": model["history"]["val_acc"][-1], "loss": model["history"]["val_loss"][-1]})
    models.append(model)
print_rows(rows, metric="accuracy")

## Results visualization

The closing figure has small-multiple output artifacts for every rung plus the metric curve from D1 to D5. The second plot shows train/validation history so local steps can be compared with generalization.

In [ ]:
closing_figure(rows, models, metric="accuracy")
plot_history(models, metric="val_acc")

## Pitfall on the hardest rung

D5 is real digits with added feature and label noise, so the pitfall has to show up in a genuinely harder training run rather than a toy picture.

In [ ]:

name, X, y = clf_digits_ladder()[-1]
bad = train_tiny_mlp(X * 25.0, y, hidden=28, epochs=22, lr=0.12, activation="sigmoid", loss_name="ce", seed=635)
good = train_tiny_mlp(X, y, hidden=28, epochs=25, lr=0.12, activation="gelu", loss_name="ce", seed=636)
print("scaled-input sigmoid D5 accuracy:", round(bad["history"]["val_acc"][-1], 3))
print("normalized-input GELU D5 accuracy:", round(good["history"]["val_acc"][-1], 3))


## Evaluate it + Practice

- Metric: inspect D5 accuracy and compare with a no-skill baseline near the majority-class rate.
- Sanity check: D1 XOR should be learnable by a hidden-layer MLP but not by one linear gate.
- Ablation: reduce width, saturate activations, or use the wrong loss and watch the metric degrade.
- Failure signals: diverging loss, flat validation curves, unstable softmax sums, or gradients with unexpected shapes.

Practice prompts:
1. Change hidden width and replot the D1–D5 curve.

2. Replace ReLU with tanh or GELU and compare D5.

3. Lower the D5 label-noise rate in `clf_digits_ladder()` and predict how the curve changes.